# TEMA 4: APLICACIONES PRÁCTICAS DE VISIÓN POR COMPUTADOR

Vamos a enfocarnos en tareas de detección y segmentación ya que clasificar imágenes es solo una parte del trabajo en visión por
computador. En muchos casos, necesitamos ir más allá: localizar objetos concretos dentro de una imagen, separarlos del fondo
o incluso diferenciar entre múltiples instancias de una misma clase.

## DETECCIÓN DE OBJETOS

La detección de objetos es uno de los pilares fundamentales de la visión por computador moderna. A diferencia de la clasificación,donde el modelo simplemente etiqueta una imagen como perteneciente a una clase, aquí el objetivo es identificar qué objetos
hay y dónde están dentro de la imagen. Esto se hace habitualmente mediante la predicción de “bounding boxes”, es decir, rectángulos
delimitadores que enmarcan la posición de cada objeto detectado junto con su clase.

### R-CNN (2014): enfoque en dos fases 

- (1) un algoritmo separado propone "regiones candidatas" donde podría haber algo,
- (2) cada región se recorta y se pasa por una CNN para clasificarla. 

Funciona, pero procesar cada región por separado es lentísimo, si hay 2000 regiones candidatas, son 2000 pasadas por la red.

### Fast/Faster R-CNN 

Integran la propuesta de regiones dentro del propio modelo (en vez de un algoritmo externo), compartiendo cómputo entre regiones. Más rápido, pero sigue siendo conceptualmente "dos fases" -> primero propongo, luego clasifico.

### SSD / YOLO 

Rompen con el esquema de dos fases. Una sola pasada por la red produce directamente todas las cajas + clases + confianzas. YOLO en concreto divide la imagen en una rejilla, y cada celda de la rejilla predice "¿hay algo aquí, qué es, y dónde exactamente dentro de esta celda?". Esto es lo que permite tiempo real, incluso sin GPU.

> El trade-off central: precisión vs velocidad-> Faster R-CNN → más preciso, más lento; YOLO/SSD → más rápido, algo menos preciso. No es "uno es mejor", es "cada uno está optimizado para un punto distinto de la curva precisión-velocidad"

> En cualquier caso, tanto YOLO como SSD, R-CNN o sus derivados se integran fácilmente en frameworks como TensorFlow, PyTorch o incluso directamente en OpenCV mediante cv2.dnn.

## SEGMENTACIÓN SEMÁNTICA

Segmentación semántica = clasificar cada píxel individualmente

### U-Net

El texto dice "forma de U" y "codificador + decodificador simétrico" pero no explica por qué esa forma. Vale la pena que lo entiendas, porque es elegante:

Codificador (la mitad bajando): es una CNN normal —> va reduciendo la resolución espacial (pooling) mientras aumenta el número de canales/features. Al final tienes un mapa muy pequeño espacialmente, pero con información muy "abstracta" (sabe qué hay, pero ha perdido precisión de dónde).
Decodificador (la mitad subiendo): hace lo contrario — va aumentando la resolución espacial de vuelta hasta el tamaño original, para producir un mapa píxel-a-píxel.
Las conexiones "skip" (la parte que hace que sea una "U" y no una "V"): en cada nivel, el decodificador recibe directamente los mapas de features del codificador en ese mismo nivel de resolución, concatenados con lo que viene de abajo.

¿Por qué importan esas conexiones skip? Porque el codificador, al reducir resolución, pierde información espacial fina (los bordes exactos de un objeto). El decodificador, sin ayuda, tendría que "reconstruir" esos bordes desde una representación muy comprimida — y lo haría mal, con bordes borrosos. Las skip connections le dan al decodificador acceso directo a la información espacial de alta resolución del codificador, en el momento exacto en que la necesita. Por eso el texto dice "combina información contextual profunda con detalles espaciales locales" — son justamente estas dos rutas (la profunda que baja y sube, y las skips que cruzan directo) trabajando juntas.

Y por eso "funciona bien con pocos datos": la arquitectura ya está diseñada para resolver el problema de "localización precisa" estructuralmente — no depende de que el modelo aprenda esa estrategia desde cero a partir de muchísimos ejemplos.

### DeepLab — la idea de "convoluciones dilatadas"

El texto menciona "convoluciones dilatadas" sin explicarlas. Una convolución normal de 3×3 mira 9 píxeles contiguos. Una convolución dilatada de 3×3 con dilatación=2 sigue teniendo 9 valores en el kernel, pero los aplica sobre píxeles espaciados (con huecos entre ellos) — por ejemplo, en lugar de mirar las posiciones (-1,0,1), mira (-2,0,2). Resultado: el kernel "ve" una región más grande de la imagen (más contexto), sin aumentar el número de parámetros ni reducir la resolución (a diferencia de pooling). Esto es lo que el texto resume como "capturar contexto global sin sacrificar resolución" — es una alternativa al pooling para conseguir "visión amplia".

## Segmentación de instancias

La limitación de 5.2 es muy concreta: si hay 3 personas, segmentación semántica las pinta todas con la misma etiqueta "persona" — los píxeles de las tres personas son indistinguibles entre sí. Segmentación de instancias añade: "y además, ¿cuál de los 3 objetos-persona es este píxel?". Es decir, combina detección (sabe que hay 3 objetos distintos, cada uno con su identidad) + segmentación (cada uno con su máscara de píxeles exacta).

### Mask R-CNN — la pieza que añade

El texto dice "extiende Faster R-CNN añadiendo una rama paralela para máscaras". La forma de pensarlo: Faster R-CNN ya te da, para cada objeto detectado, una caja + una clase. Mask R-CNN añade una tercera salida por objeto: una máscara binaria recortada a esa caja, que dice exactamente qué píxeles dentro de esa caja pertenecen al objeto y cuáles son fondo. Por eso "combina lo mejor de dos mundos" — la localización viene de la parte tipo Faster R-CNN, el detalle de píxel viene de la rama de máscaras.


### KerasCV: el "nivel de abstracción" sobre todo lo anterior
Esta sección es más fácil de entender si la ves como "lo que hace Keras para CNNs normales, pero para detección/segmentación". Igual que tf.keras.applications.EfficientNetB0(weights='imagenet') te da una arquitectura entera en una línea, KerasCV te da YOLOv8/RetinaNet/Faster R-CNN preentrenados en una línea — más utilidades específicas de detección (formato estándar de bounding boxes, augmentations que ajustan las cajas automáticamente, métrica mAP integrada).
Hay un detalle del código que vale la pena que notes, porque es exactamente el tipo de cosa que ya manejas: keras_cv.layers.RandomFlip(mode="horizontal", bounding_box_format="xywh"). En augmentation normal (como en 3.2), una transformación geométrica solo toca píxeles. Aquí, si volteas la imagen horizontalmente, las coordenadas de las bounding boxes también tienen que voltearse — si no, la caja seguiría apuntando a donde estaba el objeto antes de voltear, ahora vacío. KerasCV se encarga de esto automáticamente; en una implementación manual, este es un bug clásico y silencioso (el texto lo llama "errores de etiquetado comunes").